In [ ]:
%load_ext autoreload
%autoreload 2

from matplotlib import pyplot as plt

from privtrajgen.plot.tables import *
from privtrajgen.utils import logger, printc

In [ ]:
# Logging
log = logger.configure_root_loger(logging.INFO)


def logc(header: str, *args: Any, column_width: int = 60) -> None:
    log.info(f"{header:<{column_width}}" + "".join([str(arg) for arg in args]))

# Results

This file contains the evaluation of results and generated the paper's/thesis' figures.

## MatplotLib Configuation

Note: Matplotlib uses inches for figure sizes. The following code snippet converts the sizes to cm for easier understanding.

In [ ]:
class Mode(Enum):
    PAPER = "paper"
    THESIS = "thesis"


class Format(Enum):
    COLUMN = 1
    PAGE = 2
    THESIS = 3


def pt_to_cm(pt):
    return pt * 0.0352778


def cm_to_inch(cm):
    """Matplotlib figsize accepts inches"""
    return cm / 2.54


def pt_to_inch(pt):
    return cm_to_inch(pt_to_cm(pt))

In [ ]:
# Configuration
OUTPUT_FORMAT = "pdf"
MODE = Mode.PAPER
FORMAT = Format.COLUMN

In [ ]:
# Configure matplotlib
if MODE == Mode.PAPER:
    # ACM
    column_width_pt = 241.14749  # 241.14749pt
    page_width_pt = 506.295  # 506.295pt

    column_width = pt_to_inch(column_width_pt)
    page_width = pt_to_inch(page_width_pt)
    thesis_width = None

    fig_width = column_width if FORMAT == Format.COLUMN else page_width
    fig_height = cm_to_inch(2.5)

    FONT_SIZE = 12
    plt.rcParams.update({
        'font.size': FONT_SIZE,  # Global font size
        'axes.labelsize': FONT_SIZE,  # Increase axis labels font by 2
        'axes.titlesize': FONT_SIZE + 1,  # Set title font size to same as axis labels
        'xtick.labelsize': FONT_SIZE - 2,  # Tick-label font size
        'ytick.labelsize': FONT_SIZE - 2,  # Tick-label font size
        'legend.fontsize': FONT_SIZE,  # Make the legend/label fonts a little smaller
        'figure.subplot.wspace': 0.1,  # Reduce space between subfigures
        'figure.subplot.hspace': 0.1,  # Reduce vertical space to 0.1
        'figure.figsize': (column_width, fig_height),  # Default figure size
    })

    # Values not in rcParams
    SCATTER_SIZE = 50

elif MODE == Mode.THESIS:
    # Thesis Dimensions (Note that matplotlib uses inches, use plot.cm to convert to cm)
    LINEWIDTH_THESIS = 15.0562504028  # in cm, equals 426.79135 pt
    HALF_LINEWIDTH_THESIS = 7.3776451944  # in cm, equals 213.395675 pt
    # Thesis font sizes
    # The thesis used font 'LM Roman 10' with 10.95pt for captions and 10pt for sub-captions
    # In the default matplotlib font this equals

    FONT_SIZE = 9
    # Update all relevant font size rcParams
    plt.rcParams.update({
        'font.size': FONT_SIZE,  # Global font size
        'axes.titlesize': FONT_SIZE + 1,  # Title font size
        'axes.labelsize': FONT_SIZE,  # X and Y axis labels
        'xtick.labelsize': FONT_SIZE - 1,  # X-axis tick labels
        'ytick.labelsize': FONT_SIZE - 1,  # Y-axis tick labels
        'legend.fontsize': FONT_SIZE,  # Legend font size
        'figure.titlesize': FONT_SIZE + 1,  # Figure title font size
        'figure.labelsize': FONT_SIZE,  # Figure label font size
        'lines.markersize': 3,  # Adjust marker size
    })

    # Values not in rcParams
    SCATTER_SIZE = 30

    fig_width = cm_to_inch(LINEWIDTH_THESIS)
    aspect_ratio = 2.2
    fig_height = fig_width / aspect_ratio
else:
    raise ValueError("Invalid mode")

print(f"Width:\t\t\t{fig_width:.2f} inch\t=\t{fig_width * 2.54:.2f} cm")
print(f"Height:\t\t\t{fig_height:.2f} inch\t=\t{fig_height * 2.54:.2f} cm")
print(f"Aspect ratio:\t{fig_width / fig_height:.2f}")

## Metrics

| **ID** | **Metric**                        | **Optimal**          | **Range**     | **Description**                                               |
| ------ |-----------------------------------|----------------------|---------------|---------------------------------------------------------------|
| 1      | Jensen-Shannon Divergence (JSD)   | Smaller $\downarrow$ | $[0, 1]$      | Divergence between point distributions                        |
| 2      | Sliced Wasserstein Distance (SWD) | Smaller $\downarrow$ | $[0, \inf)$   | Distance between point distributions                          |
| 3a     | MAE (Range Query)                 | Smaller $\downarrow$ | $[0, \inf)$   | Average absolute error in range queries                       |
| 3b     | MRE (Range Query)                 | Smaller $\downarrow$ | $[0, 1]$      | Average relative error in range queries                       |
| 4      | SDC (Hotspots)                    | Larger $\uparrow$    | $[0, 1]$      | Similarity between hotspot regions                            |
| 5      | Hausdorff Distance (HD)           | Smaller $\downarrow$ | $[0, \inf)$   | Max distance between trajectories                             |
| 6      | Normalised Euclidean Distance     | Smaller $\downarrow$ | $[0, \inf)$   | Average Euclidean distance, normalised                        |
| 7      | Normalised Haversine Distance     | Smaller $\downarrow$ | $[0, \inf)$   | Average Haversine distance, normalised                        |
| 8      | Dynamic Time Warping (DTW)        | Smaller $\downarrow$ | $[0, \inf)$   | Temporal alignment distance                                   |
| 9      | JSD(TTD)                         | Smaller $\downarrow$ | $[0, 1]$      | Jensen-Shannon distance for total travelled distance patterns |
| 10     | JSD(TD)                          | Smaller $\downarrow$ | $[0, 1]$      | Jensen-Shannon distance for trajectory diameter patterns      |
| 11     | JSD(TE)                          | Smaller $\downarrow$ | $[0, 1]$      | Jensen-Shannon distance for transition patterns               |

## Print Configurations

Print the configuration table for the paper.

In [ ]:
from privtrajgen.evaluation.factories import load_runtimes


def time_to_str(time: float) -> str:
    """Convert time in seconds to a human-readable format.
    XXs for seconds, XX:XX for minutes and seconds, XX:XX:XX for hours, minutes and seconds.

    :param time: Time in seconds
    :return: Time in a human-readable format
    """
    if time < 60:
        return rf"{time:.2f}\unit{{\second}}"
    elif time < 3600:
        return rf"{int(time // 60):02d}:{int(time % 60):02d}\unit{{\minute}}"
    else:
        return rf"{int(time // 3600):02d}:{int((time % 3600) // 60):02d}:{int(time % 60):02d}\unit{{\hour}}"


def get_runtime(case: str) -> str:
    times = load_runtimes(final_evaluation[case])
    assert len(
        times) == NUM_RUNS, f"Expected {NUM_RUNS} runtimes, got {len(times)} for case {case}: {final_evaluation[case]}"
    time, h = mean_conf_half(times['runtime'])
    return time_to_str(time)


case = '1g'
get_runtime(case)

In [ ]:
import re
# Determine missing cases

missing_cases = set()

for case in final_evaluation:
    # Try porto
    try:
        get_runtime(case)
    except AssertionError:
        missing_cases.add(case)


def sort_key(case):
    match = re.match(r'(\d+)([a-zA-Z]+)', case)
    if match:
        num_part = int(match.group(1))  # Extract and convert numeric part
        str_part = match.group(2)       # Extract string part (e.g., "p")
        return (num_part, str_part)      # Sort numerically first, then alphabetically
    return (float('inf'), case)  # Handle unexpected formats (fallback)

# Remove all cases of format NUMxALPHA for NUM > 24
def select(x):
    match = re.match(r'(\d+)([a-zA-Z]+)', x)
    if match:
        num_part = int(match.group(1))
        if num_part <= 24:
            return True
        elif 'x' not in x and not (num_part>27 and 'g' in x):
            return True
        else:
            return False
    else:
        raise ValueError(f"Unexpected format: {x}")

# Only select cases that are matching the select function
missing_cases = set(x for x in missing_cases if select(x))

printc("Missing Cases: ", sorted(missing_cases, key=sort_key))
printc("Number of Missing Cases: ", len(missing_cases))


In [ ]:
from privtrajgen.config import Models

INCLUDE_X_CASES = True
THESIS=(MODE == Mode.THESIS)

tm = r"\glsshort{TM}"
HEADER_COLUMNS_ALL_RES = [
    "ID",
    # "Code",
    "Model",
    "DP-SGD",
    r"$\varepsilon_s$",
    "Cond. Info.",
    "Cond. DP.",
    r"$\varepsilon_c$",
    r"$d_{out}$",
    # "Ada",
    tm,
    "Runtime"
]
# First cc then ls
column_alignment = ['c' for _ in HEADER_COLUMNS_ALL_RES]
assert len(HEADER_COLUMNS_ALL_RES) == len(column_alignment), "Header columns and alignment must have the same length."

# Number BEFORE which to print the subheader
SUBHEADERS_ALL_RES = {
    '1': "DiffTraj",
    '9': "UNetVAE",
    '17': "UNetGAN",
    '25': r"Ablation Study I: Varying $\varepsilon_s$",
    '28': r"Ablation Study II: Varying $\varepsilon_c$",
    '32': r"Ablation Study III: Varying \gls{dp}-Conditional Information Mechanisms",
    '40': r"Non-Deep Learning Baseline: PrivTrace",
}

# Convert to list of sorted numbers, i.e., 1p, 2g, 1p becomes [1, 2]
cases = sorted(
    set(int(''.join(filter(str.isdigit, c))) for c in final_evaluation.keys() if any(ch.isdigit() for ch in c)))
# printc("List of Cases: ", cases)
# Add default rows to ablation studies (duplicates):
# 2 between 25 and 26
# 7 between 28 and 29
# 7 between 32 and 33
cases.insert(25, 2)
cases.insert(29, 7)
cases.insert(34, 7)

rows = []
for i, case in enumerate(cases):
    # Get the configuration
    config = final_evaluation[f"{case}p"]

    # Determine case ID
    # Usually, case ID = case, however, if INCLUDE_X_CASES is True, we add (a) in case the id has an x option
    if INCLUDE_X_CASES and f"{case}xp" in final_evaluation and case <= N_BASE_CASES and i <= N_BASE_CASES:
        case_id = f"{case}{highlight_x('(a)')}"
    else:
        case_id = f"{case}"

    # Determine eps_c: Usually config.cond.epsilon
    if config.model_type == Models.PrivTrace or not config.cond.dp:
        eps_c = "--"
    elif INCLUDE_X_CASES and f"{case}xp" in final_evaluation and case <= N_BASE_CASES and i <= N_BASE_CASES:
        # Return eps_c1 / eps_c2
        eps_c = f"{config.cond.epsilon} / {highlight_x(final_evaluation[f'{case}xp'].cond.epsilon)}"
    else:
        eps_c = str(config.cond.epsilon)

    # Determine eps_s
    if not config.dp.dp:
        eps_s = "--"
    else:
        eps_s = str(config.dp.target_epsilon)

    # Extract the values
    values = [
        case_id,
        # get_conf_code(config),
        get_model_str(config),
        bool_to_latex(config.dp.dp) if config.model_type != Models.PrivTrace else "--",  # PrivTrace does not use DP-SGD
        eps_s,
        get_cond_info_type(config) if isinstance(config, BaseConfig) else "--",
        get_cond_dp_str(config, thesis=THESIS) if isinstance(config, BaseConfig) else "--",
        eps_c,
        config.cond.hidden_dim if config.conditional and config.cond.dp else "--",
        # bool_to_latex(config.dp.adaptive_grad_norm),
        ballnumber(get_threat_model(config)),
        get_runtime(f"{case}p")
    ]
    # Append the values
    rows.append(values)

print(to_latex(rows, HEADER_COLUMNS_ALL_RES, column_alignment, subheaders=SUBHEADERS_ALL_RES, highlight_special=False))

## Result Table

In [ ]:
cases = sorted(
    set(int(''.join(filter(str.isdigit, c))) for c in final_evaluation.keys() if any(ch.isdigit() for ch in c)))

# Create another list with the X cases
combined_cases = (list(range(1, 9)) + [f'{i}x' for i in range(5, 9)] + list(range(9, 17)) +
                  [f'{i}x' for i in range(13, 17)] + list(range(17, 25)) + [f'{i}x' for i in range(21, 25)] +
                  list(range(25, max(cases) + 1)))

# Add default rows to ablation studies (duplicates):
# 2 between 25 and 26
# 7 between 28 and 29
# 7 between 32 and 33
cases.insert(25, 2)
cases.insert(28, 7)
cases.insert(32, 7)
# Same for combined cases
for i, case in enumerate(combined_cases):
    if case == 25:
        combined_cases.insert(i + 1, 2)
    elif case == 28:
        combined_cases.insert(i + 1, 7)
    elif case == 32:
        combined_cases.insert(i + 1, 7)


SUBHEADERS_ALL_RES = {
    1: "DiffTraj",
    9: "UNetVAE",
    17: "UNetGAN",
    25: r"Ablation Study I: Varying $\varepsilon_s$",
    28: r"Ablation Study II: Varying $\varepsilon_c$",
    32: r"Ablation Study III: Varying \gls{dp}-Conditional Information Mechanisms",
    40: r"Non-Deep Learning Baseline: PrivTrace",
    # X case lines -> No header
    '5a': None,  # Just a line without a subheader,
    '13a': None,  # Just a line without a subheader,
    '21a': None,  # Just a line without a subheader,
}
porto_subheaders = {f"{i}P": SUBHEADERS_ALL_RES[i] for i in SUBHEADERS_ALL_RES}
geo_subheaders = {f"{i}G": SUBHEADERS_ALL_RES[i] for i in SUBHEADERS_ALL_RES}

porto_cases = [f"{case}p" for case in cases]
printc("Porto Cases: ", porto_cases)

porto_cases_x = [f"{case}xp" if f"{case}xp" in final_evaluation and case <= N_BASE_CASES else f"{case}p" for case in
                 cases]
printc("Porto Cases 20: ", porto_cases_x)

porto_cases_combined = [f"{case}p" for case in combined_cases]
printc("Porto Combined Cases: ", porto_cases_combined)

# For GeoLife, remove the cases 26-37
no_gl = list(range(28, 40))
geolife_cases = [f"{case}g" for case in cases if case not in no_gl]
# Also remove the two 7 duplicates
while geolife_cases[-2] == '7g':
    geolife_cases.pop(-2)
printc("Geolife Cases: ", geolife_cases)
geolife_cases_x = [f"{case}xg" if f"{case}xg" in final_evaluation and case <= N_BASE_CASES else f"{case}g" for case in
                   cases if case not in no_gl]
printc("Geolife Cases 20: ", geolife_cases_x)
geolife_cases_combined = [f"{case}g" for case in combined_cases if case not in no_gl]
while geolife_cases_combined[-2] == '7g':
    geolife_cases_combined.pop(-2)
printc("Geolife Combined Cases: ", geolife_cases_combined)

In [ ]:
if INCLUDE_X_CASES:
    print_result_table(porto_cases_combined, subheaders=porto_subheaders)
else:
    print_result_table(porto_cases, subheaders=porto_subheaders)

In [ ]:
if INCLUDE_X_CASES:
    print_result_table(geolife_cases_combined, subheaders=geo_subheaders)
else:
    print_result_table(geolife_cases, subheaders=geo_subheaders)